In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import sklearn
from sklearn.model_selection import train_test_split
import seaborn as sns
from sklearn.model_selection import GridSearchCV
from sklearn.compose import ColumnTransformer

import lightgbm as lgb
from lightgbm import LGBMClassifier 
from sklearn.decomposition import PCA
from imblearn.pipeline import Pipeline
from sklearn.metrics import recall_score
from sklearn.metrics import make_scorer
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer, MinMaxScaler

In [60]:
df = pd.read_csv('/Users/Tudor.Orosz/Documents/School/Thesis/Extract Data Athena/Dataframe marketing and platform.csv')

In [61]:
df = df.drop(["id", "accountid", "account_sales_unit__c", "sales_unit_group__c", "name", "createddate"], axis = 1)

In [62]:
X = df.drop("opportunityid", axis = 1)
y = df.loc[:, ["opportunityid"]]

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

X_train2, X_val, y_train2, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)

In [64]:
# Define the columns to be one-hot encoded
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

X_numb = X_train.select_dtypes(include=['int64', 'float64'])

indices_pca = [i for i, col in enumerate(X.columns) if col in X_numb.columns]

In [65]:
# Define the pipeline
numeric_transformer = Pipeline(steps=[
    #('imputer', KNNImputer()),
    ('scaler', StandardScaler()), # standardizing
    # ('scaler', MinMaxScaler()), # normalizing
    #('to_df', FunctionTransformer(lambda x: pd.DataFrame(x, columns=X_train2.select_dtypes(include=['int64', 'float64']).columns)))
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, X_train2.select_dtypes(include=['int64', 'float64']).columns.tolist()),
        ('cat', categorical_transformer, categorical_cols),
        # ("pca", PCA(random_state=42, n_components=50), indices_pca),
    ])

# Add the upsampler and the pipeline to a machine learning model
model = Pipeline(steps=[
    #('oversampler', oversampler),
    ('preprocessor', preprocessor),
    ('regressor', LGBMClassifier(random_state=42, metric = "recall"))
])

parameters = {  'regressor__n_estimators': [1000], # here I changed from 1000 to 1500
    'regressor__max_depth': [5],
    'regressor__learning_rate': [0.01],
    'regressor__num_leaves': [31],
    'regressor__min_child_samples': [20],
    'regressor__reg_alpha': [0.1], 
    'regressor__reg_lambda': [0.1]
}

In [66]:
# Set up the grid search object
grid_search = GridSearchCV(model, parameters, cv=10, scoring= "recall")

In [67]:
# Fit the grid search object to the training data
grid_search.fit(X_train2, y_train2.values.ravel())

GridSearchCV(cv=10,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('scaler',
                                                                                          StandardScaler())]),
                                                                         ['M_WBN_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-Demand',
                                                                          'M_Other_Filled '
                                                                          'Out '
                                                                          'Form',
                                                                          'M_TS_Attended',
                                                                          'M_TS_Visited '
                                                                          'Booth',
                                                                          'M_LE_Attended',
                                                                          'M_WBN_Attended '
                                                                          'On-demand',
                                                                          'M_VLE_Attended',
                                                                          'M_TS_Visited '
                                                                          'booth',
                                                                          'M_SE_Att...
                                                                          24,
                                                                          25,
                                                                          26,
                                                                          27,
                                                                          28,
                                                                          29,
                                                                          30,
                                                                          31,
                                                                          32,
                                                                          33, ...])])),
                                       ('regressor',
                                        LGBMClassifier(metric='recall',
                                                       random_state=42))]),
             param_grid={'regressor__learning_rate': [0.01],
                         'regressor__max_depth': [5],
                         'regressor__min_child_samples': [20],
                         'regressor__n_estimators': [1000],
                         'regressor__num_leaves': [31],
                         'regressor__reg_alpha': [0.1],
                         'regressor__reg_lambda': [0.1]},
             scoring='recall')

In [68]:
# Get the best parameters from the grid search
best_params = grid_search.best_params_
print(best_params)

{'regressor__learning_rate': 0.01, 'regressor__max_depth': 5, 'regressor__min_child_samples': 20, 'regressor__n_estimators': 1000, 'regressor__num_leaves': 31, 'regressor__reg_alpha': 0.1, 'regressor__reg_lambda': 0.1}


In [69]:
y_pred = grid_search.predict(X_val)
y_pred

array([0, 0, 0, ..., 0, 0, 0])

In [70]:
y_proba = grid_search.predict_proba(X_val)
y_pred_new = (y_proba[:, 1] > 0.2).astype(int)

In [71]:
# Compute the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Print the confusion matrix with labels
print("Confusion matrix:")
print("           Predicted")
print("           0   1")
print("Actual 0  ", cm[0][0], " ", cm[0][1])
print("       1  ", cm[1][0], " ", cm[1][1])

Confusion matrix:
           Predicted
           0   1
Actual 0   9612   537
       1   407   397


In [72]:
# Calculate the confusion matrix
cm = confusion_matrix(y_val, y_pred_new)

# Extract the true negatives, false positives, false negatives, and true positives
tn, fp, fn, tp = cm.ravel()

# Calculate accuracy, specificity, and sensitivity
accuracy = (tp + tn) / (tp + tn + fp + fn)
specificity = tn / (tn + fp)
sensitivity = tp / (tp + fn)

# Print the results
print(f"Accuracy: {accuracy:.2f}")
print(f"Specificity: {specificity:.2f}")
print(f"Sensitivity: {sensitivity:.2f}")

Accuracy: 0.91
Specificity: 0.95
Sensitivity: 0.49
